In [1]:
from teradataml import *
#from config.settings import TD_HOST, TD_USER, TD_PASS, TD_DB

In [ ]:
# --- connect ---
#create_context(host="iteration7-w9og53takluu3v27.env.clearscape.teradata.com", username="demo_user", password="n8888888", logmech="TD2")

In [9]:
# pip install teradataml pandas

import pandas as pd
from teradataml import create_context, DataFrame
from teradataml.dataframe.copy_to import copy_to_sql
from teradataml.dataframe.sql_functions import func

# 0) Connect
create_context(
    host="iteration7-w9og53takluu3v27.env.clearscape.teradata.com",
    username="demo_user",
    password="n8888888",
    logmech="TD2",
)

# 1) Load CSV -> demo_user.test_raw  (replace if it exists)
df = pd.read_csv("data/test.csv")     # adjust path if needed

# Ensure expected columns exist; if no id column, synthesize one
if "id" not in df.columns:
    df.insert(0, "id", range(1, len(df) + 1))
# Use your real item-name column; change here if it’s named differently
ITEM_COL = "Item_Name"
assert ITEM_COL in df.columns, f"CSV must have a '{ITEM_COL}' column."

copy_to_sql(
    df,
    table_name="test_raw",
    schema_name="demo_user",
    if_exists="replace",
    index=False,
)

# 2) Build in-DB cleaning with simple functions (LOWER + TRIM + REPLACE)
tdf = DataFrame("demo_user.test_raw")  # now it exists
tdf_clean = tdf.assign(
    clean_name = func.TRIM(
        func.LOWER(
            func.REPLACE(
                func.REPLACE(
                    func.REPLACE(tdf[ITEM_COL], '-', ' '),  # "-" -> space
                    '_', ' '                                # "_" -> space
                ),
                '/', ' '                                    # "/" -> space
            )
        )
    )
)[["id", "clean_name"]]

# 3) Persist cleaned table
tdf_clean.to_sql("test_clean", schema_name="demo_user", if_exists="replace")

# 4) Quick preview
print(tdf_clean.head(10))


c:\Users\na255073\AppData\Local\Programs\Python\Python312\Lib\site-packages\teradatasqlalchemy\telemetry\queryband.py:382: UserWarning: [Teradata][teradataml](TDML_2002) Overwriting an existing context associated with Teradata Vantage Connection. Most of the operations on any teradataml DataFrames created before this will not work.
  return exposed_func(*args, **kwargs)


FileNotFoundError: [Errno 2] No such file or directory: 'data/test.csv'